# APIM ❤️ AI Gateway

## Serverless GPU lab
![flow](../../images/serverless-gpu.gif)

Deploy [SGLang](https://docs.sglang.ai/) on [Azure Container Apps](https://learn.microsoft.com/azure/container-apps/overview) with GPU acceleration (NVIDIA A100) and expose it through [Azure API Management](https://learn.microsoft.com/azure/api-management/api-management-key-concepts) as an OpenAI-compatible endpoint.

SGLang is a high-performance serving framework for large language models (LLMs). It provides an OpenAI-compatible API out of the box, making it a drop-in replacement for any OpenAI client. By running it on Azure Container Apps with GPU workload profiles, you get serverless GPU inference without managing infrastructure.

**important**

For T4 GPU, you can use models that would fit into the T4 16GB VRAM (avail mem=16.10GB)

For A100, you can use models that would fit into the A100 80GB VRAM (avail mem=78.62GB)

### Prerequisites

- [Python 3.12 or later version](https://www.python.org/) installed
- [VS Code](https://code.visualstudio.com/) installed with the [Jupyter notebook extension](https://marketplace.visualstudio.com/items?itemName=ms-toolsai.jupyter) enabled
- [Python environment](https://code.visualstudio.com/docs/python/environments#_creating-environments) with the [requirements.txt](../../requirements.txt) or run `pip install -r requirements.txt` in your terminal
- [An Azure Subscription](https://azure.microsoft.com/free/) with [Contributor](https://learn.microsoft.com/en-us/azure/role-based-access-control/built-in-roles/privileged#contributor) + [RBAC Administrator](https://learn.microsoft.com/en-us/azure/role-based-access-control/built-in-roles/privileged#role-based-access-control-administrator) or [Owner](https://learn.microsoft.com/en-us/azure/role-based-access-control/built-in-roles/privileged#owner) roles
- [Azure CLI](https://learn.microsoft.com/cli/azure/install-azure-cli) installed and [Signed into your Azure subscription](https://learn.microsoft.com/cli/azure/authenticate-azure-cli-interactively)
- A [HuggingFace token](https://huggingface.co/settings/tokens) with access to gated models (e.g., Llama)

> ⚠️ **GPU Container Apps use `Consumption-GPU-NC24-A100` workload profiles which have limited regional availability and are expensive (~$4/hr). Remember to clean up resources when done.**

▶️ Click `Run All` to execute all steps sequentially, or execute them `Step by Step`...

<a id='0'></a>
### 0️⃣ Initialize notebook variables

- Resources will be suffixed by a unique string based on your subscription id.
- Adjust the `aca_location` parameter based on [GPU workload profile regional availability](https://learn.microsoft.com/azure/container-apps/workload-profiles-overview).
- Adjust the `sglang_model_path` to serve a different model from HuggingFace.
- A HuggingFace token is required for gated models like Llama. Set it via the `HF_TOKEN` environment variable or enter it below.

In [ ]:
import os, sys, json
sys.path.insert(1, '../../shared')  # add the shared directory to the Python path
import utils

deployment_name = os.path.basename(os.path.dirname(globals()['__vsc_ipynb_file__']))
resource_group_name = f"lab-{deployment_name}"
resource_group_location = "swedencentral"

# Container Apps configuration
aca_location = "swedencentral"
sglang_image = "lmsysorg/sglang:latest"

##### Others to try:
### meta-llama/Llama-3.1-8B-Instruct
### deepseek-ai/DeepSeek-R1-Distill-Qwen-32B
### Qwen/Qwen3.5-35B-A3B

sglang_model_path = "Qwen/Qwen3.5-35B-A3B"  # change to your desired model
gpu_workload_profile_type = "Consumption-GPU-NC24-A100"  ##  or 'Consumption-GPU-NC8as-T4'

# HuggingFace token for gated models (set HF_TOKEN env var or paste directly)
hf_token = os.environ.get("HF_TOKEN", "")

# APIM configuration
apim_sku = "Basicv2"
apim_subscriptions_config = [{"name": "subscription1", "displayName": "Subscription 1"}]
inference_api_path = "openai"

utils.print_ok('Notebook initialized')

<a id='1'></a>
### 1️⃣ Verify the Azure CLI and the connected Azure subscription

The following commands ensure that you have the latest version of the Azure CLI and that the Azure CLI is connected to your Azure subscription.

In [ ]:
output = utils.run("az account show", "Retrieved az account", "Failed to get the current az account")

if output.success and output.json_data:
    current_user = output.json_data['user']['name']
    tenant_id = output.json_data['tenantId']
    subscription_id = output.json_data['id']

    utils.print_info(f"Current user: {current_user}")
    utils.print_info(f"Tenant ID: {tenant_id}")
    utils.print_info(f"Subscription ID: {subscription_id}")

<a id='2'></a>
### 2️⃣ Create deployment using 🦾 Bicep

This lab uses [Bicep](https://learn.microsoft.com/azure/azure-resource-manager/bicep/overview?tabs=bicep) to declaratively define all the resources that will be deployed. The deployment includes:

- **Log Analytics Workspace** + **Application Insights** for monitoring
- **API Management** as the gateway
- **Container Apps Environment** with a GPU workload profile (NVIDIA A100)
- **Container App** running SGLang with the specified model
- **APIM Backend + API** routing to the Container App

> ⚠️ This deployment can take 10-15 minutes to allocate the needed GPU capacity. The Container App will additionally need time to pull the Docker image and download the model weights.

In [ ]:
# Create the resource group if it doesn't exist
utils.create_resource_group(resource_group_name, resource_group_location)

# Define the Bicep parameters
bicep_parameters = {
    "$schema": "https://schema.management.azure.com/schemas/2019-04-01/deploymentParameters.json#",
    "contentVersion": "1.0.0.0",
    "parameters": {
        "apimSku": {"value": apim_sku},
        "apimSubscriptionsConfig": {"value": apim_subscriptions_config},
        "inferenceAPIPath": {"value": inference_api_path},
        "acaLocation": {"value": aca_location},
        "sglangImage": {"value": sglang_image},
        "sglangModelPath": {"value": sglang_model_path},
        "huggingFaceToken": {"value": hf_token},
        "gpuWorkloadProfileType": {"value": gpu_workload_profile_type},
    }
}

# Write the parameters to the params.json file
with open('params.json', 'w') as bicep_parameters_file:
    bicep_parameters_file.write(json.dumps(bicep_parameters))

# Run the deployment
output = utils.run(f"az deployment group create --name {deployment_name} --resource-group {resource_group_name} --template-file main.bicep --parameters params.json",
    f"Deployment '{deployment_name}' succeeded", f"Deployment '{deployment_name}' failed")

<a id='3'></a>
### 3️⃣ Get the deployment outputs

Retrieve the APIM gateway URL, subscription key, and the SGLang Container App FQDN.

In [ ]:
# Obtain all of the outputs from the deployment
output = utils.run(f"az deployment group show --name {deployment_name} -g {resource_group_name}",
    f"Retrieved deployment: {deployment_name}", f"Failed to retrieve deployment: {deployment_name}")

if output.success and output.json_data:
    log_analytics_id = utils.get_deployment_output(output, 'logAnalyticsWorkspaceId', 'Log Analytics Id')
    apim_service_id = utils.get_deployment_output(output, 'apimServiceId', 'APIM Service Id')
    apim_resource_gateway_url = utils.get_deployment_output(output, 'apimResourceGatewayURL', 'APIM API Gateway URL')
    sglang_fqdn = utils.get_deployment_output(output, 'sglangFqdn', 'SGLang FQDN')

    apim_subscriptions = json.loads(utils.get_deployment_output(output, 'apimSubscriptions').replace("\'", "\""))
    for subscription in apim_subscriptions:
        subscription_name = subscription['name']
        subscription_key = subscription['key']
        utils.print_info(f"Subscription Name: {subscription_name}")
        utils.print_info(f"Subscription Key: ****{subscription_key[-4:]}")

    api_key = apim_subscriptions[0].get("key")

<a id='4'></a>
### 4️⃣ Wait for SGLang to be ready

SGLang needs to download the model weights on first startup. This can take several minutes depending on the model size. We'll poll the `/models` endpoint until the server is ready.

In [ ]:
import time, requests

sglang_health_url = f"https://{sglang_fqdn}/v1/models"
max_wait_minutes = 20
poll_interval_seconds = 30

utils.print_info(f"Waiting for SGLang to be ready at {sglang_health_url} (up to {max_wait_minutes} min)...")

start_time = time.time()
while time.time() - start_time < max_wait_minutes * 60:
    try:
        response = requests.get(sglang_health_url, timeout=10)
        if response.status_code == 200:
            models = response.json()
            utils.print_ok(f"SGLang is ready! Available models: {json.dumps(models, indent=2)}")
            break
    except requests.exceptions.RequestException:
        pass

    elapsed = int(time.time() - start_time)
    utils.print_info(f"  SGLang not ready yet... ({elapsed}s elapsed, retrying in {poll_interval_seconds}s)")
    time.sleep(poll_interval_seconds)
else:
    utils.print_error(f"SGLang did not become ready within {max_wait_minutes} minutes. Check the Container App logs.")

<a id='requests'></a>
### 🧪 Test the API using a direct HTTP call

Send a chat completion request through the APIM gateway to the SGLang backend.

Tip: Use the [tracing tool](../../tools/tracing.ipynb) to track the behavior and troubleshoot the [policy](policy.xml).

In [ ]:
import requests

url = f"{apim_resource_gateway_url}/{inference_api_path}/chat/completions"

payload = {
    "model": sglang_model_path,
    "messages": [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What is the golden ratio?"}
    ],
    "temperature": 0.7
}

headers = {
    "Content-Type": "application/json",
    "api-key": api_key
}

response = requests.post(url, headers=headers, json=payload)

if response.status_code == 200:
    result = response.json()
    utils.print_ok("Response received from SGLang via APIM:")
    print(result["choices"][0]["message"]["content"])
else:
    utils.print_error(f"Request failed with status {response.status_code}: {response.text}")

<a id='sdk'></a>
### 🧪 Test the API using the OpenAI SDK

Use the standard [OpenAI Python SDK](https://github.com/openai/openai-python) to interact with SGLang through the APIM gateway. Since SGLang is fully OpenAI-compatible, the SDK works out of the box.

In [ ]:
from openai import OpenAI
from httpx import Client

#### OpenAI SDK send its authentication header as Bearer by default, but APIM expects it as api-key, 
# so we need to create a custom http client to set the header correctly
http_client = Client(
    headers={
        "api-key": api_key
    }
)

client = OpenAI(
    base_url=f"{apim_resource_gateway_url}/{inference_api_path}",
    api_key="",   ### the api_key parameter empty because we are passing the key via the custom http client
    http_client=http_client
)

response = client.chat.completions.create(
    model=sglang_model_path,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Explain quantum computing in simple terms."}
    ],
    temperature=0.7
)

utils.print_ok("Response received from SGLang via APIM (OpenAI SDK):")
print(response.choices[0].message.content)

<a id='streaming'></a>
### 🧪 Test streaming responses

SGLang supports streaming responses. Test it through the APIM gateway using the OpenAI SDK.

In [ ]:
from openai import OpenAI

http_client = Client(
    headers={
        "api-key": api_key
    }
)

client = OpenAI(
    base_url=f"{apim_resource_gateway_url}/{inference_api_path}",
    api_key="",
    http_client=http_client
)

utils.print_ok("Streaming response from SGLang via APIM:")
stream = client.chat.completions.create(
    model=sglang_model_path,
    messages=[
        {"role": "user", "content": "Write a short poem about cloud computing."}
    ],
    temperature=0.7,
    # reasoning_effort="low",
    stream=True
)

for chunk in stream:
    if chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="", flush=True)
print()  # newline at the end

<a id='clean'></a>
### 🗑️ Clean up resources

When you're finished with the lab, you should remove all your deployed resources from Azure to avoid extra charges and keep your Azure subscription uncluttered.
Use the [clean-up-resources notebook](clean-up-resources.ipynb) for that.

> ⚠️ **GPU Container Apps are expensive. Make sure to clean up resources when you're done!**